# Kimi Attention Residuals（Block AttnRes）

源码导航：[core/residual/attn_res.py](../../../core/residual/attn_res.py) 中的 `AttnResState`、`block_attn_res`、`PreNormBlockWithAttnRes`。

Moonshot AI (2026) 提出的 AttnRes 将深度方向的固定残差求和替换为 **softmax 加权凸组合**，使深层网络能选择性读取历史子层输出，缓解 PreNorm 稀释问题。

### 1. 核心公式

$$\alpha_{i \to l} = \frac{\exp(w_l^\top \text{RMSNorm}(v_i))}{\sum_j \exp(w_l^\top \text{RMSNorm}(v_j))}, \quad h_l = \sum_i \alpha_{i \to l} v_i$$

**Block 变体**：块内标准残差求和，块间 AttnRes，内存 $O(Nd)$（$N$ 为块数）。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.residual.attn_res import (
    AttnResAggregator, AttnResState, block_attn_res,
    init_attn_res_state, update_attn_res_partial, finalize_attn_res_block,
)
from core.norm.rmsnorm import RMSNorm

### 2. Block AttnRes 聚合演示

In [ ]:
torch.manual_seed(0)
B, T, D = 1, 4, 32
emb = torch.randn(B, T, D)
state = init_attn_res_state(emb)

# 模拟两层子层输出累加
state = update_attn_res_partial(state, torch.ones(B, T, D) * 0.1)
state = update_attn_res_partial(state, torch.ones(B, T, D) * 0.2)

agg = AttnResAggregator(D)
h = agg(state)
print('aggregated:', tuple(h.shape))
print('weights sum to 1 along depth (implicit via softmax)')

### 3. 与标准残差对比

标准 PreNorm：$h_l = h_{l-1} + f_l(h_{l-1})$，历史信息经压缩传递。

AttnRes：子层输入是**所有历史输出的加权混合**，每层有独立伪查询 $w_l$ 学习权重。

---

## 延伸阅读

- Moonshot AI, *Attention Residuals* (2026). [arXiv:2603.15031](https://arxiv.org/abs/2603.15031)
- [GitHub: MoonshotAI/Attention-Residuals](https://github.com/MoonshotAI/Attention-Residuals)